# 03 — Error Analysis

Understand where Cadence fails:
- Confusion matrix
- ROC curves: Cadence vs silero-VAD
- FIR comparison bar chart
- False-positive examples: listen to mis-classified mid_thought pauses
- Confidence calibration

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score
from torch.utils.data import DataLoader

from model.cadence_model import CadenceModel, load_processor
from model.dataset import TurnDetectionDataset, collate_fn

sns.set_theme(style='darkgrid')
FIGURES = Path('figures')
FIGURES.mkdir(exist_ok=True)

LABEL2ID = {'turn_end': 0, 'mid_thought': 1}
ID2LABEL = {0: 'turn_end', 1: 'mid_thought'}

In [ ]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
processor = load_processor()
model = CadenceModel().to(device)
ckpt = torch.load('../model/checkpoints/best.pt', map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Loaded checkpoint from epoch {ckpt['epoch']} (val FIR={ckpt['val_fir']:.4f})")

In [ ]:
test_ds = TurnDetectionDataset('../data/processed/test.parquet', processor)
loader = DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=collate_fn, num_workers=4)

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for batch in loader:
        logits = model(batch['input_values'].to(device))
        probs = torch.softmax(logits, dim=-1)
        all_preds.extend(probs.argmax(-1).cpu().numpy())
        all_probs.extend(probs[:, 0].cpu().numpy())  # P(turn_end)
        all_labels.extend(batch['labels'].numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)
print(f"Test set: {len(all_labels)} samples")

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['turn_end', 'mid_thought'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Cadence — Confusion Matrix (test set)')
fig.tight_layout()
fig.savefig(FIGURES / 'confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ROC curve
y_true = (all_labels == LABEL2ID['turn_end']).astype(int)

# silero-VAD proxy: classify turn_end if gap_ms > 400
gap_ms = test_ds.df['gap_ms'].values
vad_probs = (gap_ms / gap_ms.max()).clip(0, 1)
vad_preds_bin = (gap_ms > 400).astype(int)

fig, ax = plt.subplots(figsize=(6, 5))
for name, probs in [('Cadence', all_probs), ('silero-VAD', vad_probs)]:
    fpr, tpr, _ = roc_curve(y_true, probs)
    auc = roc_auc_score(y_true, probs)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC — Turn-End Detection')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / 'roc_curve.png', dpi=150)
plt.show()

In [ ]:
# FIR comparison
mid_mask = all_labels == LABEL2ID['mid_thought']
cadence_fir = ((all_preds == LABEL2ID['turn_end']) & mid_mask).sum() / mid_mask.sum()
vad_fir = ((vad_preds_bin == 0) & mid_mask).sum() / mid_mask.sum()  # 0=turn_end

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['silero-VAD', 'Cadence'], [vad_fir, cadence_fir], color=['#ef5350', '#66bb6a'])
ax.bar_label(bars, fmt='%.3f', padding=4)
ax.set_ylabel('False Interruption Rate (lower is better)')
ax.set_title('FIR: Cadence vs silero-VAD')
ax.set_ylim(0, max(vad_fir, cadence_fir) * 1.35)
fig.tight_layout()
fig.savefig(FIGURES / 'fir_comparison.png', dpi=150)
plt.show()
print(f"FIR reduction: {(vad_fir - cadence_fir) / vad_fir:.1%}")

In [ ]:
# Confidence calibration: are high-confidence predictions more accurate?
bins = np.linspace(0, 1, 11)
bin_acc = []
bin_conf = []
for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (all_probs >= lo) & (all_probs < hi)
    if mask.sum() == 0:
        continue
    bin_acc.append((all_preds[mask] == all_labels[mask]).mean())
    bin_conf.append(all_probs[mask].mean())

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Perfect calibration')
ax.scatter(bin_conf, bin_acc, color='#42a5f5', s=60, zorder=5)
ax.plot(bin_conf, bin_acc, color='#42a5f5')
ax.set_xlabel('Mean predicted confidence')
ax.set_ylabel('Fraction correct')
ax.set_title('Confidence Calibration')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / 'calibration.png', dpi=150)
plt.show()